# Multi-Agent Competitive Price Watch

## What you will learn

In this notebook you will build a multi-agent system from scratch using only Python and the Anthropic SDK. By the end, you'll understand:

- **Agent**: An instruction + a model call. That's it! No framework needed.
- **Tool**: A capability Claude can use, like web search.
- **Sequential chaining**: One agent's output feeds into another agent's input.
- **Parallel fan-out**: Multiple agents run at the same time.
- **Fan-in**: Combine results from multiple agents into one decision.

### Quick glossary

- **System prompt**: The instruction that tells Claude its role and how to behave.
- **User message**: The question or task we send to Claude.
- **Thread**: A way to run multiple tasks at the same time in Python.
- **API**: Application Programming Interface - how our code talks to Claude's servers.

### System diagram

```
Product to watch
       |
       v
  [Amazon.in scout]
  [Flipkart scout]
  [Croma scout]
  [Reliance Digital scout]
       |
       v (all results collected)
  [Pricing Strategist]
       |
       v
  Recommendation
```

## Setup

First, we install the Anthropic SDK and check that our API key is ready.

In [1]:
# Install the Anthropic SDK
# This is a Python package that lets us talk to Claude
!pip install anthropic


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Check if the API key exists in the environment
# os.environ is like a dictionary of settings for our computer
# The Anthropic SDK automatically reads ANTHROPIC_API_KEY from here
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    print("✓ API key found")
else:
    print("✗ API key missing - set ANTHROPIC_API_KEY environment variable")

✓ API key found


**Important**: Web search must be enabled for your organization in the [Anthropic Console](https://console.anthropic.com/). Each web search is billed separately on top of normal token costs, so we keep `max_uses` small.

In [3]:
# Configuration: what model to use and what product to watch
# By defining MODEL once, students can easily swap to a different Claude model
MODEL = "claude-haiku-4-5-20251001"  # Cheapest model with web search support

# The product we're tracking across marketplaces
PRODUCT = "boAt Airdopes 141 earbuds"

# Our own current retail price in rupees
OUR_PRICE = 1299

# Toggle for testing: when False, uses mock prices instead of live web search
# This saves API costs while learning and testing
USE_LIVE_SEARCH = True

# Mock data for testing (used when USE_LIVE_SEARCH = False)
# This is a dictionary (like a lookup table) mapping marketplace names to fake prices
MOCK_PRICES = {
    "Amazon.in": "₹999 on Amazon.in (https://amazon.in/example)",
    "Flipkart": "₹1,099 on Flipkart (https://flipkart.com/example)",
    "Croma": "Price not found on Croma",
    "Reliance Digital": "₹1,199 on Reliance Digital (https://reliancedigital.in/example)"
}

print(f"Configuration loaded:")
print(f"  Model: {MODEL}")
print(f"  Product: {PRODUCT}")
print(f"  Our price: ₹{OUR_PRICE}")
print(f"  Live search: {USE_LIVE_SEARCH}")

Configuration loaded:
  Model: claude-haiku-4-5-20251001
  Product: boAt Airdopes 141 earbuds
  Our price: ₹1299
  Live search: True


## Step 1: Your first Claude call

An **API call** is when our code sends a question to Claude's servers over the internet and gets an answer back.

A **function** is a reusable piece of code. We write it once, then call it many times with different inputs.

The **system prompt** tells Claude its role ("You are a helpful assistant"). The **user message** is our question.

In [4]:
# Import the Anthropic SDK
# "import" means we load a Python package so we can use it
import anthropic

# Create a client - this is our connection to Claude
# The client reads the API key from the environment automatically
client = anthropic.Anthropic()

def ask_claude(instruction, text):
    """Send a question to Claude and get back the answer.
    
    This is our first 'agent' - just an instruction plus a model call.
    
    Args:
        instruction: What role Claude should play (the system prompt)
        text: The question or task to send to Claude (the user message)
    
    Returns:
        A string with Claude's answer
    """
    # Call Claude's API
    # model: which version of Claude to use
    # max_tokens: maximum length of the response (keeps costs low)
    # system: the instruction/role for Claude
    # messages: a list containing our question
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        system=instruction,
        messages=[{"role": "user", "content": text}]
    )
    
    # Extract just the text from the response
    # response.content is a list of blocks; we only want text blocks
    # We loop through each block, filter for type "text", and join them
    text_parts = [block.text for block in response.content if block.type == "text"]
    return "".join(text_parts)

print("✓ ask_claude function defined")

✓ ask_claude function defined


In [5]:
# Test it! Ask Claude to explain what an API is
answer = ask_claude(
    instruction="You are a teacher explaining tech concepts to beginners.",
    text="What is an API? Answer in 2 sentences."
)

print("Claude's answer:")
print(answer)

Claude's answer:
An API (Application Programming Interface) is a set of rules that allows different software programs to communicate with each other and share information. Think of it like a menu at a restaurant—you don't need to know how the kitchen works, you just order what you want, and the kitchen provides it.


## Step 2: Adding a tool (web search)

A **tool** is a capability we give to Claude. Here we enable **web search**, which lets Claude search the internet for current information.

When we add tools to the API call, Claude can choose to use them. The response may include search results alongside text.

In [6]:
def claude_web_search(query):
    """Ask Claude to search the web and return a short answer.
    
    This function either calls the real web search API (if USE_LIVE_SEARCH is True)
    or returns a mock result (if False, for testing).
    
    Args:
        query: What to search for (e.g., "price of boAt earbuds on Amazon.in")
    
    Returns:
        A string with the search result, including source URLs
    """
    # Check if we should use mock data instead of live search
    if not USE_LIVE_SEARCH:
        # Look through the mock data to find a matching marketplace
        for marketplace, mock_result in MOCK_PRICES.items():
            # If the marketplace name appears in the query, return that mock result
            if marketplace.lower() in query.lower():
                return mock_result
        return "Price not found"
    
    # Live search mode: call Claude with the web search tool enabled
    response = client.messages.create(
        model=MODEL,
        max_tokens=500,
        messages=[{"role": "user", "content": query}],
        # tools is a list of capabilities we give Claude
        tools=[{
            "type": "web_search_20250305",  # The web search tool type
            "name": "web_search",            # Required name
            "max_uses": 2                    # Limit searches to control costs
        }]
    )
    
    # Extract only the text blocks from Claude's response
    text_parts = [block.text for block in response.content if block.type == "text"]
    return "".join(text_parts)

print("✓ claude_web_search function defined")

✓ claude_web_search function defined


In [7]:
# Test web search with a simple query
result = claude_web_search(
    f"What is the current price of {PRODUCT} on Amazon.in? Report the price in ₹ and include the source URL."
)

print("Search result:")
print(result)

Search result:
I'll search for the current price of boAt Airdopes 141 earbuds on Amazon.in.Based on my search results, there are several variants of boAt Airdopes 141 available on Amazon.in with different prices:

1. **boAt Airdopes 141/8** - ₹799.00
   - Source: https://www.amazon.in/Airdopes-141-Playtime-Resistance-Bluetooth/dp/B09N3ZNHTY

2. **boAt Airdopes 141 (Original)** - ₹995.00
   - Source: https://www.amazon.in/Airdopes-141-Bluetooth-Wireless-Playtime/dp/B09N3XMZ5F

3. **boAt Airdopes 141 Gen 2** - ₹1,099.00
   - Source: https://www.amazon.in/boAt-Airdopes-141-Streaming-Bluetooth/dp/B0F8BVSK21

4. **boAt Airdopes 141 Pro** - ₹1,284.00
   - Source: https://www.amazon.in/boAt-Airdopes-141-Bluetooth-Earphones/dp/B0DGTTP1K4

The prices vary depending on which variant you're looking for. The most affordable option is the boAt Airdopes 141/8 at ₹799, while the Pro version is the most expensive at ₹1,284.


## Step 3: Building a scout agent

A **scout** is an agent with a specific job: search one marketplace for our product's price.

We create a function that takes a marketplace name and returns the price from that marketplace. This is **sequential chaining** - one function calls another.

In [8]:
def scout_marketplace(marketplace_name):
    """Search a specific marketplace for our product's price.
    
    This is a 'scout agent' - it has one job: find the price on one marketplace.
    
    Args:
        marketplace_name: e.g., "Amazon.in" or "Flipkart"
    
    Returns:
        A string with the price and source, or "price not found"
    """
    # Build the search query
    # f-strings let us insert variables into text using {variable_name}
    query = f"""Find the current price of {PRODUCT} on {marketplace_name}. 
Report the price in ₹ as found on the page. 
Include the source URL. 
If you cannot find the price, say 'price not found' instead of guessing."""
    
    # Call our web search function
    return claude_web_search(query)

print("✓ scout_marketplace function defined")

✓ scout_marketplace function defined


In [9]:
# Test a single scout
amazon_result = scout_marketplace("Amazon.in")
print("Amazon.in scout result:")
print(amazon_result)

Amazon.in scout result:
I'll search for the current price of boAt Airdopes 141 earbuds on Amazon.in.Based on the Amazon.in search results, there are multiple variants of boAt Airdopes 141 earbuds with different prices:

1. **boAt Airdopes 141 (Original)**: ₹799.00
   - Source: https://www.amazon.in/Airdopes-141-Playtime-Resistance-Bluetooth/dp/B09N3ZNHTY

2. **boAt Airdopes 141 Gen 2**: ₹1,099.00
   - Source: https://www.amazon.in/boAt-Airdopes-141-Streaming-Bluetooth/dp/B0F8BVSK21

3. **boAt Airdopes 141 Pro Buds**: ₹1,284.00
   - Source: https://www.amazon.in/boAt-Airdopes-141-Bluetooth-Earphones/dp/B0DGTTP1K4

The basic model (Airdopes 141) is the most affordable at ₹799, while the Pro variant is priced higher. Make sure to check which variant you're interested in when purchasing.


## Step 4: Running scouts in parallel

**Parallel execution** means running multiple tasks at the same time instead of one after another.

A **thread** is like a worker. We create 4 workers (threads), and each one runs a scout for a different marketplace. They all work at the same time, making the whole process much faster.

**Fan-out** = send multiple agents to do their work at once.

In [10]:
# threading is a built-in Python module for running tasks in parallel
import threading

# List of marketplaces to check
# A list in Python is written with square brackets and commas
marketplaces = ["Amazon.in", "Flipkart", "Croma", "Reliance Digital"]

# Dictionary to store results from each scout
# A dictionary is like a lookup table: {key: value}
scout_results = {}

def run_scout(marketplace):
    """Run a scout and store its result.
    
    This function is what each thread will execute.
    """
    print(f"  🔍 Scouting {marketplace}...")
    # Call the scout and save the result in our dictionary
    result = scout_marketplace(marketplace)
    scout_results[marketplace] = result
    print(f"  ✓ {marketplace} done")

print("Starting parallel scout missions...\n")

# Create a thread for each marketplace
threads = []
for marketplace in marketplaces:
    # threading.Thread creates a new worker
    # target= tells it which function to run
    # args= passes the marketplace name to the function
    thread = threading.Thread(target=run_scout, args=(marketplace,))
    threads.append(thread)  # Add to our list of workers
    thread.start()          # Start the worker

# Wait for all threads to finish
# .join() means "wait until this worker is done"
for thread in threads:
    thread.join()

print("\nAll scouts returned!\n")

# Display all results
for marketplace, result in scout_results.items():
    print(f"--- {marketplace} ---")
    print(result)
    print()

Starting parallel scout missions...

  🔍 Scouting Amazon.in...
  🔍 Scouting Flipkart...
  🔍 Scouting Croma...
  🔍 Scouting Reliance Digital...
  ✓ Reliance Digital done
  ✓ Flipkart done
  ✓ Croma done
  ✓ Amazon.in done

All scouts returned!

--- Reliance Digital ---
I'll search for the current price of boAt Airdopes 141 earbuds on Reliance Digital.Based on my search results, I found the boAt Airdopes 141 ANC on Reliance Digital:

**Price: ₹1,599.00**

**Source URL:** https://www.reliancedigital.in/product/boat-airdopes-141-anc-tws-in-ear-earbuds-with-32-db-anc-42-hrs-playback-50-ms-low-latency-beast-mode-iwp-tech-quad-mics-with-enx-asap-charge-usb-type-c-port-and-ipx5-green-lzizis-8111860

Note: This is for the boAt Airdopes 141 ANC variant (the model with Active Noise Cancellation) in green color. The MRP is ₹5,990.00 with a 73% discount applied.

--- Flipkart ---
Based on the search results, there are multiple variants of boAt Airdopes 141 on Flipkart with different prices:

1. **b

## Step 5: The Pricing Strategist (fan-in)

**Fan-in** means combining results from multiple agents into one decision.

The **Pricing Strategist** is another agent. Its job: look at all the scout results and recommend what price we should set or what promotion to run.

In [11]:
def pricing_strategist(scout_findings, our_current_price):
    """Analyze competitor prices and recommend a pricing strategy.
    
    This is the 'fan-in' agent: it takes results from all scouts and makes one decision.
    
    Args:
        scout_findings: Dictionary of {marketplace: price_info}
        our_current_price: Our current retail price in rupees
    
    Returns:
        A recommendation string
    """
    # Build a report from all scout results
    # We loop through the dictionary and format each result nicely
    report = "Competitor pricing intelligence:\n\n"
    for marketplace, finding in scout_findings.items():
        report += f"- {marketplace}: {finding}\n"
    
    # Add our current price to the report
    report += f"\nOur current price: ₹{our_current_price}"
    
    # Define the strategist's role
    instruction = """You are a pricing strategist for an e-commerce retailer. 
Analyze competitor prices and recommend a pricing action. 
Be specific: suggest an exact price or promotion. 
Keep your recommendation under 100 words."""
    
    # Ask Claude to analyze and recommend
    recommendation = ask_claude(instruction, report)
    return recommendation

print("✓ pricing_strategist function defined")

✓ pricing_strategist function defined


In [12]:
# Run the strategist with our scout results
recommendation = pricing_strategist(scout_results, OUR_PRICE)

print("=" * 60)
print("PRICING RECOMMENDATION")
print("=" * 60)
print(recommendation)
print("=" * 60)

PRICING RECOMMENDATION
**Pricing Recommendation:**

**Reduce price to ₹1,099** immediately.

**Rationale:** Flipkart's Airdopes 141 ANC at ₹999 (with bank offer) and Amazon's Gen 2 at ₹1,099 are undercutting your ₹1,299. Your current price sits between budget (₹799-₹999) and premium (₹1,599) variants without clear differentiation. At ₹1,099, you match Amazon's competitive Gen 2 pricing, improve conversion, and maintain margin while staying below Reliance Digital's ₹1,599. Launch with a limited-time "Match & Win" promotion to drive urgency.


## Summary: What we built

We created a **multi-agent system** with:

1. **Four scout agents** that run in parallel (fan-out), each searching one marketplace
2. **One strategist agent** that combines their findings (fan-in) and recommends a price
3. **Sequential chaining**: scouts call web search, strategist uses scout results
4. **Tools**: web search capability

All with **plain Python** and the **Anthropic SDK only** - no agent frameworks needed!

### Key lessons

- An agent is just an instruction + a model call
- Multi-agent = wire several of those together
- Threads let us run agents in parallel
- Tools like web search make agents more powerful

### Try it yourself

- Change `PRODUCT` to a different item
- Add more marketplaces to the list
- Modify the strategist's instruction to focus on discounts instead of price matching
- Set `USE_LIVE_SEARCH = False` to test with mock data and save API costs